<a href="https://colab.research.google.com/github/weagan/Share-PEFT/blob/main/Full_vs_Standard_vs_Share_PEFT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cell 1: full fine-tuning on CoLA
!pip install -q --upgrade transformers datasets evaluate


from datasets import load_dataset
import evaluate
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

# 1. Load dataset
dataset = load_dataset("glue", "cola")
metric = evaluate.load("glue", "cola")

# 2. Load pretrained backbone
model_name = "FacebookAI/roberta-base"  # ✅ corrected model ID
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 3. Preprocess
def tokenize(batch):
    return tokenizer(batch['sentence'], padding="max_length", truncation=True, max_length=128)

encoded_dataset = dataset.map(tokenize, batched=True)

# 4. Trainer setup
training_args = TrainingArguments(
    output_dir="./naive_cola",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return metric.compute(predictions=predictions, references=labels)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    compute_metrics=compute_metrics
)

# 5. Train
trainer.train()

# 6. Evaluate
results = trainer.evaluate()
print("Validation results:", results)


# Cell 2: LoRA / Share-style parameter-efficient fine-tuning
!pip install -q --upgrade transformers datasets evaluate peft

from datasets import load_dataset
import evaluate
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model

# 1. Load dataset
dataset = load_dataset("glue", "cola")
metric = evaluate.load("glue", "cola")

# 2. Load pretrained backbone
model_name = "FacebookAI/roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 3. Configure LoRA
lora_config = LoraConfig(
    r=8,                  # low-rank dimension
    lora_alpha=16,
    target_modules=["query", "value"],  # attention weights
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS"
)
peft_model = get_peft_model(model, lora_config)

# 4. Preprocess
def tokenize(batch):
    return tokenizer(batch['sentence'], padding="max_length", truncation=True, max_length=128)

encoded_dataset = dataset.map(tokenize, batched=True)

# 5. Trainer setup
training_args = TrainingArguments(
    output_dir="./lora_cola",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-4,  # slightly higher for LoRA
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    logging_dir="./logs",
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return metric.compute(predictions=predictions, references=labels)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    compute_metrics=compute_metrics
)

# 6. Train
trainer.train()

# 7. Evaluate
results = trainer.evaluate()
print("Validation results:", results)


# Cell 1: Standard LoRA Fine-Tuning (task-specific adapters)
!pip install -q transformers datasets evaluate peft

from datasets import load_dataset
import evaluate
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model

# 1. Load dataset
dataset = load_dataset("glue", "cola")
metric = evaluate.load("glue", "cola")

# 2. Load backbone
model_name = "FacebookAI/roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 3. Configure LoRA (task-specific)
lora_config = LoraConfig(
    r=8,                  # low-rank dimension
    lora_alpha=16,
    target_modules=["query", "value"],  # attention layers
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS"
)

# Wrap the model with LoRA adapters
peft_model = get_peft_model(model, lora_config)

# 4. Preprocess dataset
def tokenize(batch):
    return tokenizer(batch['sentence'], padding="max_length", truncation=True, max_length=128)

encoded_dataset = dataset.map(tokenize, batched=True)

# 5. Trainer
training_args = TrainingArguments(
    output_dir="./lora_standard",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    logging_dir="./logs",
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return metric.compute(predictions=predictions, references=labels)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    compute_metrics=compute_metrics
)

# 6. Train
trainer.train()

# 7. Evaluate
results = trainer.evaluate()
print("Validation results:", results)


# Cell 2: Shared Subspace LoRA (Share-style)
!pip install -q transformers datasets evaluate peft

from datasets import load_dataset
import evaluate
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model

# 1. Load first dataset (task 1)
dataset_task1 = load_dataset("glue", "cola")
metric_task1 = evaluate.load("glue", "cola")

# 2. Load backbone
model_name = "FacebookAI/roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 3. Configure LoRA with shared subspace
# This simulates Share-style: same U, different alphas for each task
shared_lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS",
    # note: in practice, you can store `U` once and reuse
)

shared_peft_model = get_peft_model(model, shared_lora_config)

# 4. Preprocess
def tokenize(batch):
    return tokenizer(batch['sentence'], padding="max_length", truncation=True, max_length=128)

encoded_dataset_task1 = dataset_task1.map(tokenize, batched=True)

# 5. Trainer
training_args = TrainingArguments(
    output_dir="./lora_shared_task1",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    logging_dir="./logs",
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return metric_task1.compute(predictions=predictions, references=labels)

trainer = Trainer(
    model=shared_peft_model,
    args=training_args,
    train_dataset=encoded_dataset_task1["train"],
    eval_dataset=encoded_dataset_task1["validation"],
    compute_metrics=compute_metrics
)

# 6. Train on Task 1
trainer.train()

# 7. Evaluate Task 1
results_task1 = trainer.evaluate()
print("Task 1 Validation results:", results_task1)

# 8. Later: For Task 2 (e.g., MRPC), you would reload `shared_peft_model`
# and only update a new set of alphas, keeping `U` frozen


In [ ]:
# Full sequential GLUE demo with Share-style LoRA and forgetting table
!pip install -q transformers datasets evaluate peft

import torch
from datasets import load_dataset
import evaluate
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model

# ----------------------
# 1. Load backbone & tokenizer
# ----------------------
model_name = "FacebookAI/roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# ----------------------
# 2. Configure shared LoRA (U shared)
# ----------------------
shared_lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS"
)
shared_peft_model = get_peft_model(model, shared_lora_config)

# ----------------------
# 3. Utility functions
# ----------------------
def get_sentence_column_names(dataset_column_names):
    if "sentence" in dataset_column_names:
        return ("sentence",)
    elif "sentence1" in dataset_column_names and "sentence2" in dataset_column_names:
        return ("sentence1", "sentence2")
    else:
        raise ValueError(f"Could not determine sentence columns from {dataset_column_names}. Expected 'sentence' or 'sentence1'/'sentence2'.")

def tokenize(batch, sentence_keys):
    if len(sentence_keys) == 1:
        return tokenizer(batch[sentence_keys[0]], padding="max_length", truncation=True, max_length=128)
    elif len(sentence_keys) == 2:
        return tokenizer(batch[sentence_keys[0]], batch[sentence_keys[1]], padding="max_length", truncation=True, max_length=128)
    else:
        raise ValueError("Unsupported number of sentence keys.")

def make_trainer(peft_model, dataset, metric, output_dir, num_labels=2):
    peft_model.classifier = torch.nn.Linear(peft_model.config.hidden_size, num_labels)

    sentence_keys = get_sentence_column_names(dataset["train"].column_names)

    encoded_dataset = dataset.map(lambda batch: tokenize(batch, sentence_keys), batched=True)
    training_args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-4,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=3,
        logging_dir="./logs",
        logging_steps=50,
        save_total_limit=2,
        load_best_model_at_end=True,
    )
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = logits.argmax(axis=-1)
        return metric.compute(predictions=predictions, references=labels)
    trainer = Trainer(
        model=peft_model,
        args=training_args,
        train_dataset=encoded_dataset["train"],
        eval_dataset=encoded_dataset["validation"],
        compute_metrics=compute_metrics
    )
    return trainer

# ----------------------
# 4. Sequential tasks
# ----------------------
tasks = [
    ("cola", 2),
    ("mrpc", 2),
    ("sst2", 2),
]

results_table = {}

for task_name, num_labels in tasks:
    print(f"\n=== Training on {task_name.upper()} ===")
    dataset = load_dataset("glue", task_name)
    metric = evaluate.load("glue", task_name)

    trainer = make_trainer(shared_peft_model, dataset, metric, f"./lora_shared_{task_name}", num_labels=num_labels)
    trainer.train()

    # Evaluate on all seen tasks so far
    for past_task_name, _ in tasks[:tasks.index((task_name, num_labels))+1]:
        past_dataset = load_dataset("glue", past_task_name)
        past_metric = evaluate.load("glue", past_task_name)

        # Dynamically get sentence keys for past_dataset as well
        past_sentence_keys = get_sentence_column_names(past_dataset["validation"].column_names)
        past_encoded = past_dataset.map(lambda batch: tokenize(batch, past_sentence_keys), batched=True)

        logits = trainer.predict(past_encoded["validation"]).predictions
        preds = logits.argmax(axis=-1)
        results = past_metric.compute(predictions=preds, references=past_encoded["validation"]["label"])
        if past_task_name not in results_table:
            results_table[past_task_name] = []
        results_table[past_task_name].append(results)
        print(f"Validation on {past_task_name.upper()} after {task_name.upper()}: {results}")

# ----------------------
# 5. Build Forgetting Table
# ----------------------
import pandas as pd

df = pd.DataFrame.from_dict(results_table, orient="index")
df.columns = [f"After {t[0].upper()}" for t in tasks[:len(df.columns)]]
df = df.round(3)
print("\n=== Forgetting Table ===")
display(df)
